In [ ]:
# ==========================================
# FASE 5: EVALUACIÓN COMPARATIVA (CRISP-DM)
# ==========================================

import pandas as pd
import numpy as np
import time
import sys
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture

# 1. Conectar con el módulo evaluador oficial del equipo
sys.path.append('../src')
try:
    from evaluador import evaluar_modelo_recomendacion
    print("✅ Módulo 'evaluador.py' importado correctamente.")
except ImportError:
    print("❌ Error: No se pudo encontrar 'evaluador.py'. Revisa la ruta '../src'.")

# 2. Cargar los datos
ruta_csv = "C:/Users/basti/proyecto_cd/data/processed/spotify_dataset_processed.csv"
try:
    df = pd.read_csv(ruta_csv)
    print(f"✅ Dataset cargado. Dimensiones: {df.shape}")
except FileNotFoundError:
    print(f"❌ Error: No se encontró el dataset en la ruta: {ruta_csv}")

# 3. Definir características acústicas espaciales
features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

✅ Módulo 'evaluador.py' importado correctamente.
✅ Dataset cargado. Dimensiones: (81343, 20)


In [ ]:
from annoy import AnnoyIndex

print("⏳ Entrenando los 4 modelos para la evaluación comparativa...\n")

# --- MODELO 1: KNN (Búsqueda Espacial Focalizada Exacta) ---
print("1️⃣ Entrenando KNN...")
knn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
knn_model.fit(df[features])

# --- MODELO 2: K-MEANS (Agrupación Estática Global) ---
print("2️⃣ Entrenando K-Means (100 clústeres)...")
kmeans = KMeans(n_clusters=100, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(X)

# --- MODELO 3: DBSCAN (Agrupación por Densidad) ---
print("3️⃣ Entrenando DBSCAN...")
modelo_dbscan = DBSCAN(eps=0.25, min_samples=10, metric='euclidean', n_jobs=-1)
df['dbscan_cluster'] = modelo_dbscan.fit_predict(X)

# --- MODELO 4: ANN (Approximate Nearest Neighbors - Annoy/Spotify) ---
print("4️⃣ Entrenando Annoy (Construyendo el bosque hiperdimensional)...")
dimensiones = len(features)
# En Annoy, 'angular' es el equivalente matemático a la Distancia del Coseno
annoy_model = AnnoyIndex(dimensiones, 'angular') 

# --- MODELO 5: GMM (Agrupación Probabilística - Soft Clustering) ---
print("5️⃣ Entrenando Gaussian Mixture Models (GMM)...")
# Usamos 'diag' para optimizar el rendimiento en memoria y velocidad
gmm = GaussianMixture(n_components=100, covariance_type='diag', random_state=42)
# fit_predict asigna la canción al clúster donde tiene la probabilidad más alta
df['cluster_gmm'] = gmm.fit_predict(X)
print("   -> GMM entrenado.")

# Añadimos los vectores uno por uno al índice de Annoy
for i in range(len(df)):
    vector = df.iloc[i][features].values
    annoy_model.add_item(i, vector)

# Construimos el modelo usando 10 árboles (Excelente balance entre velocidad y precisión)
annoy_model.build(10)

print("\n✅ ¡Los 4 modelos han sido entrenados y están listos para competir!")

⏳ Entrenando los 4 modelos para la evaluación comparativa...

1️⃣ Entrenando KNN...
2️⃣ Entrenando K-Means (100 clústeres)...
3️⃣ Entrenando DBSCAN...
4️⃣ Entrenando Annoy (Construyendo el bosque hiperdimensional)...

✅ ¡Los 4 modelos han sido entrenados y están listos para competir!


In [ ]:
def buscar_cancion_id(nombre, dataset, artista=None):
    filtro_nombre = dataset['track_name'].str.lower() == nombre.lower()
    resultado = dataset[filtro_nombre]
    
    if resultado.empty:
        print(f"❌ No se encontró: '{nombre}'")
        return None
    if artista:
        filtro_artista = resultado['artists'].str.contains(artista, case=False, na=False)
        resultado = resultado[filtro_artista]
    if len(resultado) > 1:
        print(f"⚠️ Múltiples versiones. Usando la primera: {nombre} - {resultado.iloc[0]['artists']}")
        
    return resultado.iloc[0]['track_id']

# --- Ejecutores de Modelos ---
def ejecutar_knn(track_id, dataset, features_list, top_k=5):
    inicio = time.time()
    vector_semilla = dataset[dataset['track_id'] == track_id][features_list]
    distancias, indices = knn_model.kneighbors(vector_semilla, n_neighbors=top_k + 1)
    
    # Excluimos la semilla misma (índice 0)
    vectores_recomendados = dataset.iloc[indices[0][1:]][features_list]
    fin = time.time()
    evaluar_modelo_recomendacion("K-Nearest Neighbors (KNN)", vector_semilla, vectores_recomendados, inicio, fin)

def ejecutar_kmeans(track_id, dataset, features_list, top_k=5):
    inicio = time.time()
    vector_semilla = dataset[dataset['track_id'] == track_id][features_list]
    cluster_id = dataset[dataset['track_id'] == track_id]['cluster_kmeans'].values[0]
    
    canciones_cluster = dataset[(dataset['cluster_kmeans'] == cluster_id) & (dataset['track_id'] != track_id)]
    recomendadas = canciones_cluster.sample(n=top_k, random_state=42) if len(canciones_cluster) >= top_k else canciones_cluster
    
    vectores_recomendados = recomendadas[features_list]
    fin = time.time()
    evaluar_modelo_recomendacion("K-Means (Clustering Global)", vector_semilla, vectores_recomendados, inicio, fin)

def ejecutar_dbscan(track_id, dataset, features_list, top_k=5):
    inicio = time.time()
    vector_semilla = dataset[dataset['track_id'] == track_id][features_list]
    cluster_id = dataset[dataset['track_id'] == track_id]['dbscan_cluster'].values[0]
    
    if cluster_id == -1:
        fin = time.time()
        print("\n❌ FALLO DBSCAN: La canción fue clasificada como RUIDO (-1). No hay clúster.")
        evaluar_modelo_recomendacion("DBSCAN (Ruido)", vector_semilla, pd.DataFrame(), inicio, fin)
        return
        
    canciones_cluster = dataset[(dataset['dbscan_cluster'] == cluster_id) & (dataset['track_id'] != track_id)]
    recomendadas = canciones_cluster.sample(n=top_k, random_state=42) if len(canciones_cluster) >= top_k else canciones_cluster
    
    vectores_recomendados = recomendadas[features_list]
    fin = time.time()
    evaluar_modelo_recomendacion("DBSCAN", vector_semilla, vectores_recomendados, inicio, fin)

def ejecutar_annoy(track_id, dataset, features_list, top_k=5):
    inicio = time.time()
    
    # 1. Localizar el índice numérico de la fila de la semilla
    idx_semilla = dataset.index[dataset['track_id'] == track_id].tolist()[0]
    vector_semilla = dataset.iloc[[idx_semilla]][features_list]
    
    # 2. Buscar en el bosque de Annoy (top_k + 1 para excluir la semilla)
    indices_vecinos = annoy_model.get_nns_by_item(idx_semilla, top_k + 1)
    
    # 3. Extraer los vectores recomendados (saltando el índice 0 que es la misma semilla)
    vectores_recomendados = dataset.iloc[indices_vecinos[1:]][features_list]
    
    fin = time.time()
    
    # 4. Enviar a nuestro evaluador oficial
    evaluar_modelo_recomendacion("Approximate Nearest Neighbors (Annoy - Spotify)", vector_semilla, vectores_recomendados, inicio, fin)
    
def ejecutar_gmm(track_id, dataset, features_list, top_k=5):
    inicio = time.time()
    
    # 1. Extraer vector de la semilla
    vector_semilla = dataset[dataset['track_id'] == track_id][features_list]
    
    # 2. Identificar en qué distribución gaussiana cayó la canción principal
    cluster_id = dataset[dataset['track_id'] == track_id]['cluster_gmm'].values[0]
    
    # 3. Filtrar todas las canciones de esa misma distribución
    canciones_cluster = dataset[(dataset['cluster_gmm'] == cluster_id) & (dataset['track_id'] != track_id)]
    
    # 4. Seleccionar 5 aleatorias
    if len(canciones_cluster) >= top_k:
        recomendadas = canciones_cluster.sample(n=top_k, random_state=42)
    else:
        recomendadas = canciones_cluster
        
    vectores_recomendados = recomendadas[features_list]
    fin = time.time()
    
    # 5. Enviar al evaluador
    evaluar_modelo_recomendacion("Gaussian Mixture Models (GMM - Soft Clustering)", vector_semilla, vectores_recomendados, inicio, fin)

In [ ]:
print("=" * 65)
print("🏆 ARENA DE EVALUACIÓN COMPARATIVA DE MODELOS 🏆")
print("=" * 65)

# --- CONFIGURAR LA CANCIÓN DE PRUEBA AQUÍ ---
nombre_prueba = "Me Dê Motivo" 
artista_prueba = "Criolo" 

track_id_prueba = buscar_cancion_id(nombre_prueba, df, artista_prueba)

if track_id_prueba:
    cancion_obj = df[df['track_id'] == track_id_prueba].iloc[0]
    print(f"\n🎵 SEMILLA DE PRUEBA: '{cancion_obj['track_name']}' de {cancion_obj['artists']}")
    print(f"📊 Género: {cancion_obj['track_genre']}")
    print("-" * 65)

    # Lanzamos los 4 modelos de forma consecutiva
    ejecutar_knn(track_id_prueba, df, features)
    ejecutar_annoy(track_id_prueba, df, features)
    ejecutar_kmeans(track_id_prueba, df, features)
    ejecutar_gmm(track_id_prueba, df, features)
    ejecutar_dbscan(track_id_prueba, df, features)

🏆 ARENA DE EVALUACIÓN COMPARATIVA DE MODELOS 🏆

🎵 SEMILLA DE PRUEBA: 'Me Dê Motivo' de Criolo
📊 Género: afrobeat
-----------------------------------------------------------------

=== EVALUACIÓN DEL MODELO: K-Nearest Neighbors (KNN) ===
⏱️ KPI Latencia: 0.0156 segundos
📏 KPI Distancia Acústica Promedio: 0.0007
✅ Resultado: ÉXITO (Recomendaciones altamente coherentes)


=== EVALUACIÓN DEL MODELO: Approximate Nearest Neighbors (Annoy - Spotify) ===
⏱️ KPI Latencia: 0.0045 segundos
📏 KPI Distancia Acústica Promedio: 0.0007
✅ Resultado: ÉXITO (Recomendaciones altamente coherentes)


=== EVALUACIÓN DEL MODELO: K-Means (Clustering Global) ===
⏱️ KPI Latencia: 0.0151 segundos
📏 KPI Distancia Acústica Promedio: 0.0092
✅ Resultado: ÉXITO (Recomendaciones altamente coherentes)


=== EVALUACIÓN DEL MODELO: DBSCAN ===
⏱️ KPI Latencia: 0.0170 segundos
📏 KPI Distancia Acústica Promedio: 0.1309
⚠️ Resultado: FALLO (Recomendaciones demasiado dispersas acústicamente)

